<a href="https://colab.research.google.com/github/Izzatbek2011/AutoVend/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import asyncio
import re
import io
import nest_asyncio

from aiogram import Bot, Dispatcher, F
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.types import (
    Message,
    CallbackQuery,
    InlineKeyboardButton,
    InlineKeyboardMarkup,
    ReplyKeyboardMarkup,
    KeyboardButton,
)

import PyPDF2
import docx

nest_asyncio.apply()

BOT_TOKEN = "8825713233:AAHT6kT60fnmJVuXbPAO49OCrBzYxrkZ9JM"

bot = Bot(token=BOT_TOKEN)
dp = Dispatcher()


class TestState(StatesGroup):
    entering_bulk_questions = State()
    choosing_correct = State()
    answering = State()


user_tests = {}
active_quizzes = {}


def main_menu():
    return ReplyKeyboardMarkup(
        keyboard=[
            [KeyboardButton(text="➕ Test kiritish"), KeyboardButton(text="📝 Testni boshlash")],
            [KeyboardButton(text="📊 Mening testlarim"), KeyboardButton(text="🗑 Testlarni tozalash")]
        ],
        resize_keyboard=True
    )


def correct_choice_keyboard():
    return InlineKeyboardMarkup(
        inline_keyboard=[
            [
                InlineKeyboardButton(text="A", callback_data="correct_A"),
                InlineKeyboardButton(text="B", callback_data="correct_B"),
                InlineKeyboardButton(text="C", callback_data="correct_C"),
                InlineKeyboardButton(text="D", callback_data="correct_D"),
            ]
        ]
    )


def parse_questions_from_text(text: str):
    parsed_questions = []

    # Ortiqcha bo'shliq va yangi qatorlarni bir xillashtirish
    cleaned_text = re.sub(r'\s+', ' ', text)

    # Savollarni raqamlar bo'yicha ajratish
    blocks = re.split(r'(?=\b\d+[\.\)]\s*)', cleaned_text)

    for block in blocks:
        block = block.strip()
        if not block:
            continue

        pattern = (
            r'(?:^\d+[\.\)]\s*)?(?P<q>.+?)'
            r'\s+[Aa][\)\.\:\-]?\s*(?P<a>.+?)'
            r'\s+[Bb][\)\.\:\-]?\s*(?P<b>.+?)'
            r'\s+[Cc][\)\.\:\-]?\s*(?P<c>.+?)'
            r'\s+[Dd][\)\.\:\-]?\s*(?P<d>.+)'
        )
        match = re.search(pattern, block)

        if match:
            q_data = match.groupdict()
            question_text = q_data["q"].strip()
            ans_a = q_data["a"].strip()
            ans_b = q_data["b"].strip()
            ans_c = q_data["c"].strip()
            ans_d_full = q_data["d"].strip()

            correct_match = re.search(r'(?:Javob|To\'g\'ri javob|Kalit)[\s\:\-]*([A-Da-d])', ans_d_full, re.IGNORECASE)

            correct_ans = None
            if correct_match:
                correct_ans = correct_match.group(1).upper()
                ans_d = re.sub(r'(?:Javob|To\'g\'ri javob|Kalit)[\s\:\-]*[A-Da-d].*', '', ans_d_full, flags=re.IGNORECASE).strip()
            else:
                ans_d = ans_d_full

            parsed_questions.append({
                "question": question_text,
                "A": ans_a,
                "B": ans_b,
                "C": ans_c,
                "D": ans_d,
                "correct": correct_ans
            })

    return parsed_questions


async def extract_text_from_document(message: Message) -> str:
    document = message.document
    file_name = document.file_name.lower()

    file_bytes = io.BytesIO()
    await bot.download(document, destination=file_bytes)
    file_bytes.seek(0)

    text = ""

    if file_name.endswith('.txt'):
        text = file_bytes.read().decode('utf-8', errors='ignore')

    elif file_name.endswith('.pdf'):
        pdf_reader = PyPDF2.PdfReader(file_bytes)
        for page in pdf_reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"

    elif file_name.endswith('.docx'):
        doc = docx.Document(file_bytes)
        text = "\n".join([p.text for p in doc.paragraphs])

    return text


async def send_question_with_timer(user_id, state: FSMContext):
    quiz = active_quizzes.get(user_id)
    if not quiz:
        return

    current_idx = quiz["current"]
    questions = quiz["questions"]

    if current_idx >= len(questions):
        await show_final_results(user_id, state)
        return

    q = questions[current_idx]

    keyboard = InlineKeyboardMarkup(
        inline_keyboard=[
            [InlineKeyboardButton(text=f"A) {q['A']}", callback_data="ans_A")],
            [InlineKeyboardButton(text=f"B) {q['B']}", callback_data="ans_B")],
            [InlineKeyboardButton(text=f"C) {q['C']}", callback_data="ans_C")],
            [InlineKeyboardButton(text=f"D) {q['D']}", callback_data="ans_D")],
        ]
    )

    msg = await bot.send_message(
        user_id,
        f"📌 <b>{current_idx + 1}-savol / {len(questions)}</b>\n\n"
        f"❓ {q['question']}\n\n"
        f"⏱ <b>Vaqt: 60 sekund</b>",
        reply_markup=keyboard,
        parse_mode="HTML"
    )

    quiz["timer_task"] = asyncio.create_task(question_timer(user_id, current_idx, msg.message_id, state))


async def question_timer(user_id, question_idx, message_id, state: FSMContext):
    await asyncio.sleep(60)

    quiz = active_quizzes.get(user_id)
    if quiz and quiz["current"] == question_idx:
        quiz["wrong"] += 1
        quiz["current"] += 1

        try:
            await bot.edit_message_text(
                chat_id=user_id,
                message_id=message_id,
                text="⏰ <b>Vaqt tugadi!</b> Javob berilmadi.",
                parse_mode="HTML"
            )
        except Exception:
            pass

        await asyncio.sleep(1)
        await send_question_with_timer(user_id, state)


async def show_final_results(user_id, state: FSMContext):
    quiz = active_quizzes.get(user_id)
    if not quiz:
        return

    total = len(quiz["questions"])
    correct = quiz["correct"]
    wrong = quiz["wrong"]
    percentage = round((correct / total) * 100) if total > 0 else 0

    await bot.send_message(
        user_id,
        f"🏁 <b>TEST YAKUNLANDI!</b>\n\n"
        f"📊 Jami savollar: {total} ta\n"
        f"✅ To'g'ri javoblar: {correct} ta\n"
        f"❌ Noto'g'ri/Vaqt o'tdi: {wrong} ta\n"
        f"📈 Natija: <b>{percentage}%</b>",
        reply_markup=main_menu(),
        parse_mode="HTML"
    )

    active_quizzes.pop(user_id, None)
    await state.clear()


@dp.message(Command("start"))
async def start_cmd(message: Message, state: FSMContext):
    await state.clear()
    await message.answer(
        "👋 <b>Xush kelibsiz!</b>\n\n"
        "Menyu orqali test kiritishingiz va ularni yechishingiz mumkin.",
        reply_markup=main_menu(),
        parse_mode="HTML"
    )


@dp.message(F.text == "➕ Test kiritish")
async def add_test_start(message: Message, state: FSMContext):
    await state.set_state(TestState.entering_bulk_questions)
    await message.answer(
        "📝 <b>Testlarni har qanday ko'rinishda yuboring:</b>\n"
        "- Qatorma-qator yoki bir qatorda yozilgan matn\n"
        "- TXT, PDF, DOCX fayllari\n\n"
        "Bot barchasini o'zi ajratib oladi.",
        reply_markup=main_menu(),
        parse_mode="HTML"
    )


@dp.message(TestState.entering_bulk_questions, F.content_type.in_({'text', 'document'}))
async def process_bulk_input(message: Message, state: FSMContext):
    user_id = message.from_user.id
    raw_text = ""

    if message.document:
        status_msg = await message.answer("📄 Fayl o'qilmoqda...")
        raw_text = await extract_text_from_document(message)
        await status_msg.delete()
    else:
        raw_text = message.text.strip()

    if not raw_text:
        await message.answer("❌ Matn ajratib bo'lmadi.")
        return

    parsed = parse_questions_from_text(raw_text)

    if not parsed:
        await message.answer(
            "❌ <b>Testlar topilmadi.</b>\n\n"
            "Savollar va 4 ta variant (A, B, C, D) mavjudligini tekshiring.",
            parse_mode="HTML"
        )
        return

    if user_id not in user_tests:
        user_tests[user_id] = []

    ready_questions = [q for q in parsed if q["correct"] is not None]
    pending_questions = [q for q in parsed if q["correct"] is None]

    user_tests[user_id].extend(ready_questions)

    if pending_questions:
        await state.update_data(pending=pending_questions, pending_idx=0)
        await state.set_state(TestState.choosing_correct)

        q = pending_questions[0]
        await message.answer(
            f"❓ <b>Savol:</b> {q['question']}\n\n"
            f"A) {q['A']}\n"
            f"B) {q['B']}\n"
            f"C) {q['C']}\n"
            f"D) {q['D']}\n\n"
            "✅ <b>Ushbu savolning to'g'ri javobini tanlang:</b>",
            reply_markup=correct_choice_keyboard(),
            parse_mode="HTML"
        )
    else:
        await state.clear()
        total_q = len(user_tests[user_id])
        await message.answer(
            f"✅ <b>Barcha {len(ready_questions)} ta savol muvaffaqiyatli saqlandi!</b>\n\n"
            f"Jami saqlangan testlar: <b>{total_q} ta</b>",
            reply_markup=main_menu(),
            parse_mode="HTML"
        )


@dp.callback_query(TestState.choosing_correct, F.data.startswith("correct_"))
async def process_correct_choice(callback: CallbackQuery, state: FSMContext):
    correct_ans = callback.data.split("_")[1]
    data = await state.get_data()
    user_id = callback.from_user.id

    pending = data["pending"]
    idx = data["pending_idx"]

    q = pending[idx]
    q["correct"] = correct_ans
    user_tests[user_id].append(q)

    idx += 1

    if idx < len(pending):
        await state.update_data(pending_idx=idx)
        next_q = pending[idx]
        await callback.message.edit_text(
            f"❓ <b>Savol:</b> {next_q['question']}\n\n"
            f"A) {next_q['A']}\n"
            f"B) {next_q['B']}\n"
            f"C) {next_q['C']}\n"
            f"D) {next_q['D']}\n\n"
            "✅ <b>Ushbu savolning to'g'ri javobini tanlang:</b>",
            reply_markup=correct_choice_keyboard(),
            parse_mode="HTML"
        )
    else:
        await state.clear()
        total_q = len(user_tests[user_id])
        await callback.message.edit_text(
            f"✅ <b>Barcha savollar saqlandi!</b>\n\n"
            f"Jami saqlangan testlar: <b>{total_q} ta</b>",
            parse_mode="HTML"
        )
        await callback.message.answer("Testlarni boshlashingiz mumkin.", reply_markup=main_menu())

    await callback.answer()


@dp.message(F.text == "📊 Mening testlarim")
async def show_my_tests(message: Message):
    questions = user_tests.get(message.from_user.id, [])
    await message.answer(
        f"📋 Sizda jami <b>{len(questions)} ta</b> saqlangan savol bor.",
        reply_markup=main_menu(),
        parse_mode="HTML"
    )


@dp.message(F.text == "🗑 Testlarni tozalash")
async def clear_tests(message: Message):
    user_tests[message.from_user.id] = []
    await message.answer("🗑 Siz kiritgan barcha testlar o'chirildi.", reply_markup=main_menu())


@dp.message(F.text == "📝 Testni boshlash")
async def start_quiz(message: Message, state: FSMContext):
    user_id = message.from_user.id
    questions = user_tests.get(user_id, [])

    if not questions:
        await message.answer(
            "❌ Hali hech qanday test kiritmadingiz. Avval <b>➕ Test kiritish</b> tugmasini bosing.",
            parse_mode="HTML"
        )
        return

    active_quizzes[user_id] = {
        "questions": questions,
        "current": 0,
        "correct": 0,
        "wrong": 0,
        "timer_task": None
    }

    await state.set_state(TestState.answering)
    await message.answer("🚀 <b>Test boshlanmoqda! Har bir savolga 1 minut vaqt beriladi.</b>", parse_mode="HTML")
    await asyncio.sleep(1)
    await send_question_with_timer(user_id, state)


@dp.callback_query(TestState.answering, F.data.startswith("ans_"))
async def process_answer(callback: CallbackQuery, state: FSMContext):
    user_id = callback.from_user.id
    selected_ans = callback.data.split("_")[1]

    quiz = active_quizzes.get(user_id)
    if not quiz:
        await callback.answer("Test topilmadi.", show_alert=True)
        return

    if quiz.get("timer_task"):
        quiz["timer_task"].cancel()

    current_idx = quiz["current"]
    q = quiz["questions"][current_idx]
    correct_ans = q["correct"]

    if selected_ans == correct_ans:
        quiz["correct"] += 1
        res_text = f"✅ <b>To'g'ri javob!</b>\n\nSiz tanladingiz: {selected_ans}"
    else:
        quiz["wrong"] += 1
        res_text = f"❌ <b>Noto'g'ri javob!</b>\n\nSiz tanladingiz: {selected_ans}\nTo'g'ri javob: <b>{correct_ans}</b>"

    quiz["current"] += 1

    await callback.message.edit_text(res_text, parse_mode="HTML")
    await callback.answer()

    await asyncio.sleep(1.5)
    await send_question_with_timer(user_id, state)


async def main():
    print("BOT ISHGA TUSHDI...")
    await dp.start_polling(bot)


if __name__ == "__main__":
    asyncio.run(main())

BOT ISHGA TUSHDI...
